# Kynera — Example Notebook

This notebook demonstrates the full workflow of the **Kynera** library:
downloading, loading, extracting, converting, computing derived variables,
validating against observations, and plotting ERA5 reanalysis data.

**Course**: Geospatial Processing 2025/2026 — Politecnico di Milano  

---

## Table of Contents
1. [Setup & imports](#1-setup)
2. [Download ERA5 data](#2-download)
3. [Load the dataset](#3-load)
4. [Explore the dataset](#4-explore)
5. [Extract at stations](#5-extract)
6. [Convert units](#6-units)
7. [Compute derived variables](#7-derived)
8. [Validate against observations](#8-validate)
9. [Plot spatial maps](#9-plot)

## 1. Setup & imports <a id='1-setup'></a>

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # add repo root to path

import kynera
import pandas as pd
import matplotlib.pyplot as plt

print(f"Kynera version: {kynera.__version__}")

## 2. Download ERA5 data <a id='2-download'></a>

You need a Copernicus CDS account and API key.  
Register at: https://cds.climate.copernicus.eu

> **Skip this cell** if you already have the sample file in `DATA/`.

In [ ]:
CDS_KEY     = "your-api-key-here"   # replace with your key
OUTPUT_PATH = "DATA/era5_adriatico_sample.zip"

os.makedirs("DATA", exist_ok=True)

kynera.download_era5(
    variables=[
        '2m_temperature',
        'mean_sea_level_pressure',
        'total_precipitation',
        '10m_u_component_of_wind',
        '10m_v_component_of_wind',
        '2m_dewpoint_temperature',
    ],
    year='2025',
    months='10',
    days=['25', '26', '27'],
    times=['00:00', '06:00', '12:00', '18:00'],
    area=[46.5, 12.0, 39.0, 20.0],   # [N, W, S, E] — Adriatic Sea
    output_path=OUTPUT_PATH,
    cds_key=CDS_KEY,
)

## 3. Load the dataset <a id='3-load'></a>

Kynera automatically handles both plain `.nc` files and the new CDS Beta `.zip` format
(which contains separate files for instant and accumulated variables).

In [ ]:
ZIP_PATH = "DATA/era5_adriatico_sample.zip"

ds = kynera.load_era5(ZIP_PATH)
print(ds)

## 4. Explore the dataset <a id='4-explore'></a>

In [ ]:
print("Variables:", list(ds.data_vars))
print("Dimensions:", dict(ds.dims))
print("Latitude range: ", float(ds.latitude.min()),  "→", float(ds.latitude.max()))
print("Longitude range:", float(ds.longitude.min()), "→", float(ds.longitude.max()))

In [ ]:
# Quick overview of raw values
import numpy as np

for var in ds.data_vars:
    vals = ds[var].values.flatten()
    vals = vals[~np.isnan(vals)]
    print(f"{var:6s}  min={vals.min():.2f}  max={vals.max():.2f}  mean={vals.mean():.2f}")

## 5. Extract at stations <a id='5-extract'></a>

Define a list of in-situ stations within the ERA5 domain and extract
the nearest grid-point values for every timestep.

In [ ]:
stations = pd.DataFrame({
    "station_id":   ["S001",    "S002",    "S003"],
    "station_name": ["Trieste", "Ancona",  "Bari"],
    "lat":          [45.65,     43.60,     41.12],
    "lon":          [13.76,     13.50,     16.87],
})

df_raw = kynera.extract_at_stations(ds, stations)
print(f"Extracted {len(df_raw)} rows for {df_raw['station_id'].nunique()} stations")
df_raw.head(6)

## 6. Convert units <a id='6-units'></a>

Convert raw CDS units to human-readable values:
K → °C, Pa → hPa, m → mm

In [ ]:
df = kynera.convert_units(df_raw)
df[['station_name', 'valid_time', 't2m_celsius', 'mslp_hpa', 'tp_mm']].head(9)

In [ ]:
# Time series of temperature per station
fig, ax = plt.subplots(figsize=(11, 4))
for name, grp in df.groupby("station_name"):
    ax.plot(grp["valid_time"], grp["t2m_celsius"], marker="o", label=name)
ax.set_ylabel("Temperature (°C)")
ax.set_title("ERA5 — 2m Temperature at selected stations")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Compute derived variables <a id='7-derived'></a>

Kynera computes additional meteorological variables from ERA5 base fields:
- **Wind speed** and **wind direction** from u/v components
- **Relative humidity** from 2m temperature and dewpoint (Magnus formula)

In [ ]:
df = kynera.compute_derived(df)

derived_cols = [c for c in ['wind_speed_10m', 'wind_dir_10m', 'rh_2m'] if c in df.columns]
print("Derived columns available:", derived_cols)
df[['station_name', 'valid_time'] + derived_cols].head(9)

In [ ]:
# Plot relative humidity if available
if 'rh_2m' in df.columns:
    fig, ax = plt.subplots(figsize=(11, 4))
    for name, grp in df.groupby("station_name"):
        ax.plot(grp["valid_time"], grp["rh_2m"], marker="s", label=name)
    ax.set_ylabel("Relative Humidity (%)")
    ax.set_ylim(0, 110)
    ax.set_title("ERA5 — 2m Relative Humidity at selected stations")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Validate against observations <a id='8-validate'></a>

Compare ERA5 values against in-situ observations to assess ground-truth quality.
This is particularly useful when using ERA5 as reference for weather forecasting.

Here we simulate observational data. Replace `df_obs` with your real station CSVs.

In [ ]:
import numpy as np

# Simulate observations: ERA5 + realistic noise + small bias per station
rng = np.random.default_rng(seed=42)
bias_per_station = {"S001": -0.8, "S002": +0.3, "S003": +1.2}

obs_rows = []
for _, row in df.iterrows():
    if pd.notna(row.get("t2m_celsius")):
        noise = rng.normal(0, 0.5)
        bias  = bias_per_station.get(row["station_id"], 0)
        obs_rows.append({
            "station_id": row["station_id"],
            "valid_time": row["valid_time"],
            "temp_obs":   row["t2m_celsius"] + bias + noise,
        })
df_obs = pd.DataFrame(obs_rows)
df_obs.head()

In [ ]:
stats = kynera.validate_vs_observations(
    df_era5        = df,
    df_obs         = df_obs,
    variable_era5  = "t2m_celsius",
    variable_obs   = "temp_obs",
)

print("Per-station validation metrics:")
print(stats.to_string(index=False))

In [ ]:
# Bar chart of RMSE per station
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, metric, color in zip(axes, ['bias', 'mae', 'rmse'], ['steelblue', 'orange', 'tomato']):
    axes_labels = stats['station_id']
    ax.bar(axes_labels, stats[metric], color=color, alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(metric.upper())
    ax.set_ylabel('°C')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle("ERA5 vs Observations — Temperature Error Metrics", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Plot spatial maps <a id='9-plot'></a>

Visualize ERA5 fields on a georeferenced map using Cartopy.

In [ ]:
# Map of 2m temperature at the first timestep
fig = kynera.plot_field(ds, variable='t2m', time_index=0, cmap='RdYlBu_r')
plt.show()

In [ ]:
# Map of total precipitation
fig = kynera.plot_field(ds, variable='tp', time_index=0, cmap='Blues')
plt.show()

In [ ]:
# Map of mean sea level pressure
fig = kynera.plot_field(ds, variable='msl', time_index=0, cmap='coolwarm')
plt.show()